### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:llama-3.3-70b-versatile")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content="Parrots are known for their remarkable ability to mimic human speech and other sounds, but why do they do it? The answer lies in their evolution, social behavior, and communication needs.\n\nIn the wild, parrots use vocalizations to communicate with each other for various purposes, such as:\n\n1. **Contact calls**: To maintain contact with their flock members, especially when foraging or flying.\n2. **Alarm calls**: To warn others of potential predators or threats.\n3. **Courtship and mating**: To attract a mate or strengthen pair bonds.\n4. **Territorial defense**: To defend their territory from other parrots or birds.\n\nWhen parrots are kept as pets, they may learn to mimic human speech and other sounds for several reasons:\n\n1. **Social interaction**: Parrots are highly social animals and may learn to talk to interact with their human caregivers, seeking attention, affection, or rewards.\n2. **Imitation**: Parrots are known for their exceptional mimicry abilitie

In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get weather of a location"""
    return f"It's sunny in {location}"

model_with_tools=model.bind_tools([get_weather])

In [4]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    #View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'tool_calls': [{'id': 'cdxwmxz2v', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 218, 'total_tokens': 232, 'completion_time': 0.03669594, 'completion_tokens_details': None, 'prompt_time': 0.012991662, 'prompt_tokens_details': None, 'queue_time': 0.056428987, 'total_time': 0.049687602}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019fbe2c-8efe-71d2-a90e-9d3b1e6b9ca5-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'cdxwmxz2v', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 218, 'output_tokens': 14, 'total_tokens': 232}
Tool: get_weather
Args: {'location': 'Boston'}


### Tool Execution Loops

In [5]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

I'm glad you're interested in the weather! However, I must correct myself - I don't have have access to real-time information. If you would like to know the current weather in Boston, I can suggest checking a weather website or app for the most up-to-date information.


In [6]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'jr11ap117', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 217, 'total_tokens': 231, 'completion_time': 0.035706396, 'completion_tokens_details': None, 'prompt_time': 0.020930376, 'prompt_tokens_details': None, 'queue_time': 0.161715632, 'total_time': 0.056636772}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fbe30-9542-7902-8a30-8ddf405f79dd-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'jr11ap117', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 217, 'output_tokens': 14, 'total_tokens': 231}),
 ToolMessage(content="It's